# Learning to Make a Quadruped Robot Walk

In this notebook, we train a simulated **Unitree Go2-style quadruped robot** to walk forward using reinforcement learning.

The robot is simulated in **MuJoCo**, a physics engine commonly used for robotics and control. The learning algorithm is **Proximal Policy Optimization (PPO)**, a popular reinforcement learning method for continuous-control tasks.

The goal is simple to state:

> Learn a controller that makes the robot move forward along the x-axis without falling.

Although this sounds simple, quadruped walking is a difficult control problem. The robot must coordinate many joints, interact with the ground through contacts, maintain balance, and produce useful forward motion at the same time.

<img src="figures/ppo_go2.jpg" alt="Go2 MuJoCo" width="400"/>

## The Robot

The robot has a floating base and four legs.

Each leg has three actuated joints:

- hip abduction/adduction,
- hip flexion/extension,
- knee flexion/extension.

Since the robot has four legs, the policy controls **12 actuated joints** in total.

The action produced by the policy is therefore a 12-dimensional vector:

```python
action_dim = 12
```

Each action value affects one joint.

## Why Walking Is Difficult

Quadruped locomotion is challenging for several reasons.

First, the robot has a **floating base**. The body is not directly controlled. The policy can only move the joints, and the body moves because the feet push against the ground.

Second, walking depends on **contacts**. Feet repeatedly touch and leave the ground. Contact dynamics are discontinuous and can make the learning signal noisy.

Third, the robot must **balance while moving**. A policy that moves the legs aggressively may generate forward motion, but it may also make the robot fall.

Fourth, the action space is high-dimensional. The policy must coordinate 12 joints in a rhythmic and stable way.

Finally, the reward is indirect. The policy is not told exactly how each leg should move. It only receives feedback about the resulting behavior, such as forward progress and body stability.

## Gait Prior

Learning quadruped locomotion from scratch can be very difficult. At the beginning of training, the robot usually does not know how to move its legs in a useful rhythm.

To make learning easier, we use a simple hand-designed gait prior.

The gait prior creates a rough trot-like pattern:

- front-left and rear-right legs move together,
- front-right and rear-left legs move together,
- each leg alternates between stance and swing.

During the stance phase, the foot stays low and moves backward relative to the body.

During the swing phase, the foot moves forward and lifts off the ground.

The gait prior does not solve the task by itself. It only gives the robot a useful starting pattern. The policy then learns residual corrections on top of it.

## Observations

At every step, the policy receives an observation describing the current state of the robot.

The observation contains information such as:

- the robot body height and orientation,
- joint positions,
- joint velocities,
- base velocities,
- a simple gait phase signal.

The global x/y position of the robot is removed from the observation. This is useful because we do not want the policy to memorize absolute positions in the world. Instead, we want it to learn a reusable walking behavior.

A phase signal is also included:

```python
phase_obs = [
    sin(2 * pi * gait_phase),
    cos(2 * pi * gait_phase),
]
```

This tells the policy where it is in the walking cycle.

## Actions

The policy outputs one action per actuated joint.

However, the action is not used directly as torque. Instead, the action modifies a desired joint position:

```python
q_des = default_joint_positions + gait_prior + action_scale * action
```

This desired joint position is then tracked using a PD controller.

This makes the learning problem easier. Instead of learning raw torque control from scratch, the policy learns how to adjust a reasonable target pose.

## PD Control

A PD controller converts desired joint positions into torques:

```python
torque = kp * (q_des - q) - kd * qdot
```

where:

- `q_des` is the desired joint position,
- `q` is the current joint position,
- `qdot` is the current joint velocity,
- `kp` controls how strongly the joint moves toward the target,
- `kd` adds damping.

The PD controller provides low-level stabilization. PPO only needs to learn the higher-level corrections.

## Residual Learning

The policy learns a residual action around the default pose and gait prior:

```python
q_des = default_joint_positions + gait_prior + action_scale * action
```

This means the policy does not need to invent walking completely from scratch.

Instead, it learns how to improve the gait by adjusting joint targets.

For example, the policy may learn to:

- increase or decrease step size,
- stabilize the body,
- compensate for imperfect contact timing,
- recover from small disturbances,
- avoid falling,
- make the gait more efficient.

This approach is called **residual learning** because the neural network learns corrections on top of an existing controller or motion pattern.

## Reward Function

The reward function tells the robot what behavior we want.

In this task, the reward encourages the robot to:

- move forward along the x-axis,
- stay alive,
- keep the body at a reasonable height,
- keep the body upright,
- avoid using excessive torque,
- avoid deviating too much from the desired joint targets.

A simplified reward has the form:

```python
reward = (
    forward_reward
    + alive_bonus
    - base_height_penalty
    - orientation_penalty
    - torque_penalty
    - joint_target_penalty
)
```

The most important term is the forward progress reward:

```python
forward_progress = x_after - x_before
```

This rewards the robot for moving forward during the current environment step.

The other terms help prevent undesirable solutions, such as falling, jumping too high, flipping over, or using unnecessarily large torques.

## Why We Use PPO

PPO is a reinforcement learning algorithm that works well for continuous-control tasks.

It learns a policy by repeatedly:

1. running the current policy in the environment,
2. collecting observations, actions, rewards, and value estimates,
3. estimating which actions were better or worse than expected,
4. updating the policy using the collected data,
5. repeating the process.

PPO is designed to update the policy gradually. This is important in locomotion tasks because very large policy updates can easily destroy a partially working gait.

## Actor-Critic Policy

The policy network is an actor-critic model.

The **actor** chooses actions. In this task, it outputs a distribution over 12-dimensional continuous actions.

The **critic** estimates how good the current state is. This value estimate helps PPO decide whether the actions taken were better or worse than expected.

Using both an actor and a critic usually makes learning more stable than using rewards alone.

## What We Expect to See During Training

At the beginning of training, the robot may:

- fall quickly,
- slide,
- shake,
- move its legs without useful coordination,
- exploit parts of the reward in unexpected ways.

As training improves, the robot should start to:

- stay upright for longer,
- produce more regular leg motion,
- move forward along the x-axis,
- use smoother actions,
- maintain a more stable body height.

The final gait does not need to be perfect or biologically realistic. The goal of this notebook is educational: to show how simulation, PD control, gait priors, reward design, and PPO can be combined to train a basic quadruped locomotion policy.

## Summary

This notebook combines several important robotics and reinforcement learning ideas:

- physics simulation with MuJoCo,
- quadruped locomotion,
- PD joint control,
- gait priors,
- residual learning,
- reward design,
- PPO training,
- actor-critic neural networks.

Together, these components allow us to train a simulated quadruped robot to learn a forward walking behavior.

### A Little Motivation

In [ ]:
from IPython.display import Video
Video("figures/ppo_good.mp4", embed=True)

### Now let's install MuJoCo

In [ ]:
# Install MuJoCo
%pip install mujoco

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl" # We need egl for off-screen rendering
import numpy as np
import torch

from ppo import PPO, RolloutBuffer
from quadruped_env import QuadrupedEnv
from video import record_policy_video

## Setting Up the Reinforcement Learning Pipeline

Training a quadruped robot to walk requires more than just defining an environment and a policy. We need a complete reinforcement learning pipeline that connects simulation, data collection, policy updates, logging, checkpointing, and video recording.

This section explains how the full training loop is organized.

## Simulation Model

We first specify the MuJoCo XML file that describes the robot and the scene:

```python
xml_path = "unitree_go2/scene.xml"
```

This file contains the robot model, joints, actuators, geometry, ground plane, and other simulation settings.

The environment loads this XML file and creates a MuJoCo simulation.

## Training Duration

```python
total_steps = 2_000_000
```

This is the total number of environment steps used for training.

One environment step does not necessarily mean one MuJoCo physics step. In this notebook, each environment step internally runs several MuJoCo steps using `frame_skip`.

## Rollout Length

```python
rollout_steps = 4096
```

PPO does not update the policy after every single environment step.

Instead, it first collects a batch of experience called a rollout. The rollout stores:

- observations,
- actions,
- rewards,
- done flags,
- log probabilities,
- value estimates.

Once `4096` steps have been collected, PPO uses this data to update the actor-critic network.

This is called on-policy learning, because PPO updates the policy using data collected from the current version of the policy.

## Episode Length

```python
episode_length = 1000
```

An episode is one attempt of the robot starting from an initial state.

The episode ends if:

- the robot falls,
- the robot becomes unstable,
- the maximum episode length is reached.

Here, the maximum episode length is 1000 environment steps.

## Frame Skip

```python
frame_skip = 5
```

The policy does not choose a new action at every MuJoCo physics step.

Instead, the same action is applied for multiple physics steps. With `frame_skip = 5`, each policy action is held constant for 5 MuJoCo simulation steps.

This has two benefits:

1. it makes control smoother,
2. it reduces the number of decisions the policy needs to make.

The MuJoCo timestep is `0.002`, then one policy step corresponds to:

```python
0.002 * 5 = 0.01 seconds
```

So the policy acts at approximately 100 Hz.

## Saving and Video Recording

```python
save_interval = 50_000
video_interval = 10_000
```

During training, we periodically save checkpoints and record videos.

Checkpoints allow us to reload the policy later:

```text
checkpoints/ppo_go2_step_50000.pt
```

Videos allow us to visually inspect the behavior of the robot:

```text
videos/ppo_go2_step_10000.mp4
```

This is important because reward values alone do not always tell us whether the robot is learning the desired behavior. A policy may receive higher reward while still moving in an unnatural or unstable way.

## Choosing the Device

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
```

If a GPU is available, PyTorch uses CUDA for neural network computation. Otherwise, it runs on the CPU.

The MuJoCo simulation itself still runs separately from the neural network computation. Although there are variants (e.g. `mjx`) that can run directly on the GPU!

## Action Scale

```python
action_scale = 0.25
```

The policy outputs actions in a normalized range.

These actions are multiplied by `action_scale` before being added to the desired joint positions:

```python
q_des = default_joint_positions + gait_prior + action_scale * action
```

A smaller action scale makes learning safer and more stable, but it gives the policy less control authority.

A larger action scale gives the policy more freedom, but it can also make the robot unstable.

In [ ]:
xml_path = "unitree_go2/scene.xml"

total_steps = 2_000_000
rollout_steps = 4096

episode_length = 1000
frame_skip = 5

save_interval = 50_000
video_interval = 10_000

device = "cuda" if torch.cuda.is_available() else "cpu"

action_scale = 0.25

In order to be able to track and monitor the training procedure, we initialize 2 different enviroments.

## Training Environment

```python
env = QuadrupedEnv(
    xml_path=xml_path,
    episode_length=episode_length,
    frame_skip=frame_skip,
    action_scale=action_scale,
)
```

This is the main environment used for training.

The agent interacts with this environment to collect experience.

At each step:

1. the policy receives an observation,
2. the policy outputs an action,
3. the environment simulates the robot,
4. the environment returns the next observation, reward, done flag, and diagnostic information.

## Video Environment

```python
video_env = QuadrupedEnv(
    xml_path=xml_path,
    episode_length=episode_length,
    frame_skip=frame_skip,
    action_scale=action_scale,
)
```

A second environment is created for video recording.

This is useful because recording video can interfere with or slow down the main training loop. By using a separate environment, we can evaluate and visualize the policy without disturbing the training environment state.

## PPO Agent

```python
agent = PPO(
    obs_dim=env.obs_dim,
    action_dim=env.action_dim,
    device=device,
    lr=5e-4,
    gamma=0.98,
    gae_lambda=0.95,
    clip_eps=0.2,
    value_coef=0.5,
    entropy_coef=0.005,
    max_grad_norm=0.5,
    ppo_epochs=5,
    minibatch_size=256,
)
```

The PPO agent contains the actor-critic neural network and the optimizer.

The actor chooses actions.

The critic estimates the value of each state.

The main PPO hyperparameters are:

| Hyperparameter | Meaning |
|---|---|
| `lr` | learning rate for the neural network |
| `gamma` | discount factor for future rewards |
| `gae_lambda` | smoothing factor for advantage estimation |
| `clip_eps` | limits how much the policy can change in one update |
| `value_coef` | weight of the critic loss |
| `entropy_coef` | encourages exploration |
| `max_grad_norm` | clips large gradients |
| `ppo_epochs` | number of optimization passes per rollout |
| `minibatch_size` | number of samples used in each gradient update |

## Rollout Buffer

```python
buffer = RolloutBuffer(
    obs_dim=env.obs_dim,
    action_dim=env.action_dim,
    rollout_steps=rollout_steps,
    device=device,
)
```

The rollout buffer stores the experience collected by the current policy.

For every environment step, it stores:

```text
obs
action
log_prob
reward
done
value
```

When the buffer is full, PPO uses the stored data to compute advantages and update the policy.

After the update, the buffer is cleared and data collection begins again.

In [ ]:
env = QuadrupedEnv(
    xml_path=xml_path,
    episode_length=episode_length,
    frame_skip=frame_skip,
    action_scale=action_scale,
)

video_env = QuadrupedEnv(
    xml_path=xml_path,
    episode_length=episode_length,
    frame_skip=frame_skip,
    action_scale=action_scale,
)

agent = PPO(
    obs_dim=env.obs_dim,
    action_dim=env.action_dim,
    device=device,
    lr=5e-4,
    gamma=0.98,
    gae_lambda=0.95,
    clip_eps=0.2,
    value_coef=0.5,
    entropy_coef=0.005,
    max_grad_norm=0.5,
    ppo_epochs=5,
    minibatch_size=256,
)

buffer = RolloutBuffer(
    obs_dim=env.obs_dim,
    action_dim=env.action_dim,
    rollout_steps=rollout_steps,
    device=device,
)

# Gait Prior: Math and Intuition

In this notebook, the quadruped robot is not asked to discover walking completely from scratch. Instead, we provide a simple **gait prior**: a hand-designed periodic leg motion that gives the robot a rough walking rhythm.

The reinforcement learning policy then learns **residual corrections** on top of this prior.

The high-level idea is:

$$
q_{\mathrm{des}} = q_{\mathrm{default}} + q_{\mathrm{gait}}(t) + \alpha a_t
$$

where:

- $q_{\mathrm{des}}$ is the desired joint position sent to the PD controller,
- $q_{\mathrm{default}}$ is a nominal standing pose,
- $q_{\mathrm{gait}}(t)$ is the time-dependent gait prior,
- $a_t$ is the action produced by the policy,
- $\alpha$ is the action scale.

So the gait prior provides a structured motion, while the policy learns how to modify it.

## Why Use a Gait Prior?

Learning locomotion from scratch is difficult because the robot must discover several things at the same time:

- how to move each leg,
- when each foot should touch the ground,
- how to balance the body,
- how to generate forward motion,
- how to avoid falling.

A gait prior reduces the difficulty by giving the robot a reasonable initial stepping pattern.

Instead of learning the whole behavior from zero, PPO learns corrections around a useful motion template.

This is called **residual learning**.

## Periodic Gait Phase

Walking is periodic. Each leg repeatedly goes through a cycle of stance and swing.

We define a phase variable:

$$
\phi(t) = (f t + \phi_0) \bmod 1
$$

where:

- $t$ is the simulation time,
- $f$ is the gait frequency,
- $\phi_0$ is the phase offset for a particular leg,
- $\phi(t) \in [0, 1)$ is the normalized phase of the leg.

In the code:

```python
phase = (freq * t + phases[leg_name]) % 1.0
```

The frequency controls how quickly the stepping pattern repeats.

For example, if:

$$
f = 1.4
$$

then each gait cycle takes:

$$
T = \frac{1}{f} \approx 0.714 \text{ seconds}
$$

## Trot Gait Phase Offsets

The gait prior uses a simple trot-like pattern.

In a trot, diagonal legs move together:

- front-left and rear-right are in phase,
- front-right and rear-left are in phase,
- the two diagonal pairs are half a cycle apart.

The phase offsets are:

$$
\phi_{0,FL} = 0
$$

$$
\phi_{0,RR} = 0
$$

$$
\phi_{0,FR} = 0.5
$$

$$
\phi_{0,RL} = 0.5
$$

In code:

```python
phases = {
    "FL": 0.0,
    "FR": 0.5,
    "RL": 0.5,
    "RR": 0.0,
}
```

This means that when the front-left and rear-right legs are in one part of the gait cycle, the front-right and rear-left legs are halfway through the opposite part of the cycle.

## Stance and Swing

Each leg alternates between two phases:

1. **stance phase**: the foot is on or near the ground and pushes the body forward,
2. **swing phase**: the foot lifts and moves forward for the next step.

We define a duty factor $d$:

$$
d \in (0, 1)
$$

The duty factor is the fraction of the cycle spent in stance.

In this notebook:

$$
d = 0.5
$$

So each leg spends half of the cycle in stance and half in swing.

The phase rule is:

$$
\text{stance if } \phi < d
$$

$$
\text{swing if } \phi \ge d
$$

In code:

```python
if phase < duty:
    # stance
else:
    # swing
```

## Foot Trajectory During Stance

During stance, the foot stays low and moves backward relative to the robot body.

This backward motion helps produce forward motion of the body through contact with the ground.

We define a local stance parameter:

$$
s = \frac{\phi}{d}
$$

where:

$$
s \in [0, 1]
$$

The desired foot position in the sagittal plane is:

$$
x(s) = x_{\mathrm{nom}} + L\left(\frac{1}{2} - s\right)
$$

$$
z(s) = z_{\mathrm{nom}}
$$

where:

- $x_{\mathrm{nom}}$ is the nominal foot x-position,
- $z_{\mathrm{nom}}$ is the nominal foot height,
- $L$ is the step length,
- $s$ moves from 0 to 1 during stance.

At the beginning of stance:

$$
s = 0 \Rightarrow x = x_{\mathrm{nom}} + \frac{L}{2}
$$

At the end of stance:

$$
s = 1 \Rightarrow x = x_{\mathrm{nom}} - \frac{L}{2}
$$

So the foot moves backward relative to the body.

In code:

```python
s = phase / duty
x = nominal_x[leg_type] + step_length * (0.5 - s)
z = nominal_z
```

## Foot Trajectory During Swing

During swing, the foot moves forward and lifts off the ground.

We define a swing parameter:

$$
s = \frac{\phi - d}{1 - d}
$$

where:

$$
s \in [0, 1]
$$

The x-position moves from back to front:

$$
x(s) = x_{\mathrm{nom}} + L\left(s - \frac{1}{2}\right)
$$

The z-position follows a smooth bump:

$$
z(s) = z_{\mathrm{nom}} + H \sin(\pi s)
$$

where:

- $H$ is the step height,
- $\sin(\pi s)$ is zero at the beginning and end of swing,
- $\sin(\pi s)$ is maximum at the middle of swing.

At the beginning of swing:

$$
s = 0 \Rightarrow z = z_{\mathrm{nom}}
$$

At the middle of swing:

$$
s = 0.5 \Rightarrow z = z_{\mathrm{nom}} + H
$$

At the end of swing:

$$
s = 1 \Rightarrow z = z_{\mathrm{nom}}
$$

This gives a simple smooth foot-lifting motion.

In code:

```python
s = (phase - duty) / (1.0 - duty)
x = nominal_x[leg_type] + step_length * (s - 0.5)
z = nominal_z + step_height * np.sin(np.pi * s)
```

## Complete Foot Trajectory

Combining stance and swing, the desired foot trajectory is piecewise:

$$
(x(\phi), z(\phi)) =
\begin{cases}
\left(
    x_{\mathrm{nom}} + L\left(\frac{1}{2} - \frac{\phi}{d}\right),
    z_{\mathrm{nom}}
\right), & \phi < d \\
\left(
    x_{\mathrm{nom}} + L\left(\frac{\phi - d}{1 - d} - \frac{1}{2}\right),
    z_{\mathrm{nom}} + H \sin\left(\pi \frac{\phi - d}{1 - d}\right)
\right), & \phi \ge d
\end{cases}
$$

This defines a simple periodic stepping motion for each foot.

## From Foot Position to Joint Angles

The gait prior first defines where the foot should be. However, the robot is controlled through joint angles.

So we need to convert desired foot positions into joint targets.

This is done using a simple planar two-link inverse kinematics model.

For each leg, we approximate the thigh and calf as two links of length:

$$
l_1 = 0.213
$$

$$
l_2 = 0.213
$$

Given a desired foot position $(x, z)$ relative to the hip, the distance from hip to foot is:

$$
r = \sqrt{x^2 + z^2}
$$

In code:

```python
d = np.sqrt(x * x + z * z)
```

The distance is clipped so that the target remains reachable:

$$
r \leftarrow \mathrm{clip}(r, r_{\min}, l_1 + l_2 - \epsilon)
$$

## Knee Angle from the Law of Cosines

The knee angle is computed using the law of cosines.

For a triangle with sides $l_1$, $l_2$, and $r$:

$$
\cos(\theta_k) = \frac{l_1^2 + l_2^2 - r^2}{2 l_1 l_2}
$$

Then:

$$
\theta_k = \arccos\left(\frac{l_1^2 + l_2^2 - r^2}{2 l_1 l_2}\right)
$$

The robot's calf joint convention uses a negative angle when the knee is bent, so the target calf angle is:

$$
q_{\mathrm{calf}} = -(\pi - \theta_k)
$$

In code:

```python
cos_knee = (l1 * l1 + l2 * l2 - d * d) / (2.0 * l1 * l2)
cos_knee = np.clip(cos_knee, -1.0, 1.0)

knee_internal = np.arccos(cos_knee)
calf = -(np.pi - knee_internal)
```

## Thigh Angle

The thigh angle is computed from two parts.

First, we compute the direction from the hip to the foot:

$$
\alpha = \arctan2(-x, -z)
$$

Then we compute the triangle angle at the hip:

$$
\beta = \arccos\left(\frac{l_1^2 + r^2 - l_2^2}{2 l_1 r}\right)
$$

The desired thigh angle is:

$$
q_{\mathrm{thigh}} = \alpha + \beta
$$

In code:

```python
alpha = np.arctan2(-x, -z)

cos_hip = (l1 * l1 + d * d - l2 * l2) / (2.0 * l1 * d)
cos_hip = np.clip(cos_hip, -1.0, 1.0)

beta = np.arccos(cos_hip)

thigh = alpha + beta
```

For simplicity, the hip abduction target is set to zero:

$$
q_{\mathrm{hip}} = 0
$$

This means the prior only creates a sagittal-plane stepping motion.

## Gait Offsets Around the Default Pose

The inverse kinematics gives absolute joint targets for each leg:

$$
q_{\mathrm{IK}} =
\begin{bmatrix}
q_{\mathrm{hip}} \\
q_{\mathrm{thigh}} \\
q_{\mathrm{calf}}
\end{bmatrix}
$$

But the environment stores the gait prior as an offset around the default pose:

$$
q_{\mathrm{gait}} = q_{\mathrm{IK}} - q_{\mathrm{default}}
$$

In code:

```python
gait[start_idx + 0] = hip - self.default_joint_positions[start_idx + 0]
gait[start_idx + 1] = thigh - self.default_joint_positions[start_idx + 1]
gait[start_idx + 2] = calf - self.default_joint_positions[start_idx + 2]
```

This makes the final desired joint position:

$$
q_{\mathrm{des}} = q_{\mathrm{default}} + q_{\mathrm{gait}} + \alpha a_t
$$

Since:

$$
q_{\mathrm{gait}} = q_{\mathrm{IK}} - q_{\mathrm{default}}
$$

we get:

$$
q_{\mathrm{des}} = q_{\mathrm{IK}} + \alpha a_t
$$

So the policy learns corrections around the IK-generated gait.

## Residual Policy Action

The policy action $a_t$ is normalized, usually with each component in approximately:

$$
a_t \in [-1, 1]
$$

The action is scaled by $\alpha$:

$$
\Delta q_t = \alpha a_t
$$

where $\alpha$ is `action_scale`.

For example, if:

$$
\alpha = 0.25
$$

then each action can modify a joint target by up to approximately:

$$
\pm 0.25 \text{ radians}
$$

This keeps the learned corrections moderate. The gait prior produces the basic stepping pattern, and the policy fine-tunes it.

## PD Tracking of the Desired Joint Angles

Once $q_{\mathrm{des}}$ is computed, the robot tracks it with a PD controller:

$$
\tau = k_p(q_{\mathrm{des}} - q) - k_d \dot{q}
$$

where:

- $\tau$ is the torque applied to the joints,
- $q$ is the current joint angle,
- $\dot{q}$ is the current joint velocity,
- $k_p$ is the proportional gain,
- $k_d$ is the derivative gain.

This means the policy does not directly output torques. It outputs corrections to desired joint positions.

The PD controller handles low-level stabilization.

## Summary

The gait prior works in four steps:

1. Compute a periodic phase for each leg:

$$
\phi(t) = (ft + \phi_0) \bmod 1
$$

2. Use the phase to define a stance or swing foot trajectory:

$$
(x(\phi), z(\phi))
$$

3. Convert the desired foot position into joint angles using inverse kinematics:

$$
(x, z) \rightarrow (q_{\mathrm{hip}}, q_{\mathrm{thigh}}, q_{\mathrm{calf}})
$$

4. Add the policy's residual action:

$$
q_{\mathrm{des}} = q_{\mathrm{default}} + q_{\mathrm{gait}}(t) + \alpha a_t
$$

The result is a controller that combines:

- a simple hand-designed walking rhythm,
- low-level PD control,
- reinforcement learning corrections.

This makes the locomotion problem easier and more stable than learning raw torques from scratch.


In [ ]:
class PolicyWrapper:
    """
    Adapter so the old video recorder can call select_action(obs, noise_std=0.0).
    """
    def __init__(self, ppo_agent):
        self.ppo_agent = ppo_agent

    def select_action(self, obs, noise_std=0.0):
        obs_t = torch.tensor(
            obs,
            dtype=torch.float32,
            device=self.ppo_agent.device,
        ).unsqueeze(0)

        with torch.no_grad():
            mu, std, value = self.ppo_agent.policy(obs_t)

        return mu.cpu().numpy()[0]

## Output Folders

```python
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("videos", exist_ok=True)
```

These folders are created before training starts.

The `checkpoints` folder stores saved models.

The `videos` folder stores recorded policy rollouts.

## Resetting the Environment

```python
obs = env.reset()
```

Before training starts, the environment is reset to an initial robot configuration.

The initial observation is stored in `obs`.

This observation is the first input to the policy.

## Episode Statistics

```python
episode_return = 0.0
episode_length_counter = 0
episode_number = 0
episode_start_x = env.data.qpos[0].copy()
```

These variables track what happens during each episode.

`episode_return` stores the sum of rewards during the episode.

`episode_length_counter` counts how many steps the episode lasted.

`episode_number` counts how many episodes have finished.

`episode_start_x` stores the robot's x-position at the beginning of the episode, so we can measure how far it moved.

## Selecting an Action

```python
action, log_prob, value = agent.select_action(obs)
```

The current observation is passed to the policy.

The agent returns:

- an action,
- the log probability of that action,
- the critic's value estimate.

The action is used to control the robot.

The log probability and value estimate are stored because PPO needs them later during the update.

## Stepping the Environment

```python
next_obs, reward, done, info = env.step(action)
```

The action is applied in the environment.

The environment returns:

| Variable | Meaning |
|---|---|
| `next_obs` | observation after applying the action |
| `reward` | reward received for this step |
| `done` | whether the episode ended |
| `info` | diagnostic information such as position and velocity |

This is where MuJoCo advances the robot simulation.

## Storing Experience

```python
buffer.add(
    obs=obs,
    action=action,
    log_prob=log_prob,
    reward=reward,
    done=float(done),
    value=value,
)
```

The transition is stored in the rollout buffer.

This stored data will later be used for the PPO update.

PPO needs the old log probabilities because it compares the new policy against the policy that generated the data.

## Handling Episode Termination

```python
if done:
```

If the episode ends, we compute useful diagnostics.

The total forward displacement is:

```python
episode_dx = info["x_position"] - episode_start_x
```

The simulated episode time is:

```python
sim_time = episode_length_counter * env.model.opt.timestep * env.frame_skip
```

The average forward velocity is:

```python
avg_vx = episode_dx / max(sim_time, 1e-6)
```

These metrics help us understand whether the robot is actually moving forward.

After logging, the environment is reset:

```python
obs = env.reset()
```

The episode statistics are also reset.

## Training the Policy

```python
if buffer.is_full():
```

When the rollout buffer reaches `rollout_steps`, we update the PPO agent.

First, we estimate the value of the final observation:

```python
last_value = agent.value(obs)
```

This is needed for bootstrapping. Bootstrapping allows PPO to estimate the return beyond the end of the rollout if the episode has not terminated.

Then we update the policy:

```python
latest_losses = agent.update(buffer, last_value)
```

Inside this update, PPO:

1. computes advantages,
2. computes returns,
3. splits the rollout into minibatches,
4. performs several epochs of gradient descent,
5. updates the actor and critic networks.

After the update, the buffer is cleared:

```python
buffer.clear()
```

## PPO Loss Logging

After each PPO update, we print:

```text
policy_loss
value_loss
entropy
approx_kl
```

These values help diagnose training.

| Metric | What it tells us |
|---|---|
| `policy_loss` | whether the actor is improving according to the PPO objective |
| `value_loss` | how well the critic predicts returns |
| `entropy` | how random or exploratory the policy is |
| `approx_kl` | how much the policy changed during the update |

If `approx_kl` becomes very large, the policy may be changing too aggressively.

If entropy becomes very small too early, the policy may stop exploring.

If value loss is very large, the critic may be struggling to predict returns.

## Saving Checkpoints

```python
if step % save_interval == 0:
    checkpoint_path = f"checkpoints/ppo_go2_step_{step}.pt"
    agent.save(checkpoint_path)
```

The model is saved every `save_interval` steps.

Checkpoints are useful because they allow us to:

- resume training,
- compare policies from different training stages,
- keep a backup if later training becomes unstable,
- use the trained policy later for evaluation.

## Recording Videos

```python
if step % video_interval == 0:
    video_path = f"videos/ppo_go2_step_{step}.mp4"
    _, r = record_policy_video(
        env=video_env,
        agent=PolicyWrapper(agent),
        output_path=video_path,
        duration=10.0,
        fps=30,
    )
```

Videos are recorded periodically to visualize the learned behavior.

This is especially important in robotics. A reward curve may improve even if the robot is learning an undesirable behavior, such as hopping, dragging its body, or exploiting the simulator.

By watching videos during training, we can check whether the robot is learning a reasonable walking gait.

## Full Pipeline Summary

The complete reinforcement learning pipeline is:

1. Load the MuJoCo robot model.
2. Create the training environment.
3. Create a separate environment for video recording.
4. Create the PPO agent.
5. Create the rollout buffer.
6. Reset the environment.
7. Repeatedly collect experience from the current policy.
8. Store each transition in the rollout buffer.
9. When the buffer is full, update the PPO policy.
10. Periodically save model checkpoints.
11. Periodically record videos.
12. Continue until the total number of training steps is reached.

## Why the Pipeline Is Complicated

This pipeline is more complicated than a simple supervised learning loop because reinforcement learning has several interacting parts.

In supervised learning, we usually have a fixed dataset.

In reinforcement learning, the dataset is created by the current policy while it interacts with the environment. As the policy changes, the data distribution also changes.

This means training involves both:

- collecting new experience,
- updating the policy from that experience.

For robotics tasks, the difficulty increases further because:

- simulation must be stable,
- actions must be physically meaningful,
- rewards must encourage the right behavior,
- the policy must balance exploration and stability,
- contacts with the ground create noisy dynamics,
- a bad policy can quickly make the robot fall.

The training loop therefore needs to carefully coordinate simulation, learning, logging, saving, and evaluation.

## What to Watch During Training

Useful signs of progress include:

- episode return increasing,
- episode length increasing,
- forward displacement `dx` becoming positive,
- average forward velocity `avg_vx` becoming positive,
- the robot falling less often,
- videos showing more stable movement.

However, no single metric is perfect.

For locomotion, always inspect both the numerical logs and the videos.


In [ ]:
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("videos", exist_ok=True)

obs = env.reset()

episode_return = 0.0
episode_length_counter = 0
episode_number = 0
episode_start_x = env.data.qpos[0].copy()

latest_losses = {}

for step in range(1, total_steps + 1):
    action, log_prob, value = agent.select_action(obs)

    next_obs, reward, done, info = env.step(action)

    buffer.add(
        obs=obs,
        action=action,
        log_prob=log_prob,
        reward=reward,
        done=float(done),
        value=value,
    )

    obs = next_obs
    episode_return += reward
    episode_length_counter += 1

    if done:
        episode_dx = info["x_position"] - episode_start_x

        sim_time = (
            episode_length_counter
            * env.model.opt.timestep
            * env.frame_skip
        )

        avg_vx = episode_dx / max(sim_time, 1e-6)

        print(
            f"Episode {episode_number:05d} | "
            f"Step {step:08d} | "
            f"Return {episode_return:8.2f} | "
            f"Length {episode_length_counter:4d} | "
            f"dx {episode_dx:+.3f} | "
            f"avg_vx {avg_vx:+.3f} | "
            f"mean|a| {info['mean_abs_action']:.3f} | "
            f"KL {latest_losses.get('approx_kl', 0.0):.4f}"
        )

        obs = env.reset()

        episode_return = 0.0
        episode_length_counter = 0
        episode_number += 1
        episode_start_x = env.data.qpos[0].copy()

    if buffer.is_full():
        last_value = agent.value(obs)
        latest_losses = agent.update(buffer, last_value)
        buffer.clear()

        print(
            f"[PPO Update] Step {step:08d} | "
            f"policy_loss {latest_losses['policy_loss']:.4f} | "
            f"value_loss {latest_losses['value_loss']:.4f} | "
            f"entropy {latest_losses['entropy']:.4f} | "
            f"KL {latest_losses['approx_kl']:.4f}"
        )

    if step % save_interval == 0:
        checkpoint_path = f"checkpoints/ppo_go2_step_{step}.pt"
        agent.save(checkpoint_path)
        print(f"[Checkpoint] Saved {checkpoint_path}")

    if step % video_interval == 0:
        video_path = f"videos/ppo_go2_step_{step}.mp4"
        _, r = record_policy_video(
            env=video_env,
            agent=PolicyWrapper(agent),
            output_path=video_path,
            duration=10.0,
            fps=30,
        )
        print(f"[Video] Saved {video_path}. Reward: {r:.2f}")